# 04 · Validacion territorial de huecos H1

Este notebook toma los huecos H1 detectados con persistencia homologica y los valida territorialmente. La idea central es separar tres cosas que geometricamente pueden verse parecidas:

1. Huecos relevantes para cobertura publica de salud.
2. Huecos esperados o de baja prioridad porque caen sobre parques, suelo no habitacional, areas verdes o zonas poco pobladas.
3. Huecos intermedios que requieren revision manual.

Un hueco H1 no es automaticamente una falta de cobertura. La persistencia mide robustez geometrica, no necesidad social. Por eso cruzamos cada hueco con poblacion, densidad, rezago social y uso no habitacional. El resultado final es una clasificacion semiautomatica: 80% basada en datos geoespaciales y 20% reservada para revision manual de casos dudosos.

## Dependencias, rutas y parametros

Los umbrales estan al inicio para que el criterio sea transparente y facil de modificar. El notebook intenta usar capas locales si existen en `data/raw` o `data/external` y, si faltan, deja columnas nulas o de respaldo sin detener todo el flujo.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

try:
    import folium
except ImportError:
    folium = None

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
EXTERNAL_DIR = DATA_DIR / "external"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MAPS_DIR = PROJECT_ROOT / "reports" / "maps"

for directory in [RAW_DIR, EXTERNAL_DIR, PROCESSED_DIR, FIGURES_DIR, MAPS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CRS_METRICO = "EPSG:32614"
CRS_WEB = "EPSG:4326"

PCT_NO_HABITACIONAL_UMBRAL = 0.50
POBLACION_BAJA_UMBRAL = 500
QUANTILE_PRIORIDAD = 0.75
PCT_NO_HAB_DUDOSO = 0.40

HUECOS_CSV = PROCESSED_DIR / "huecos_h1_significativos.csv"
HUECOS_CLASIFICADOS_CSV = PROCESSED_DIR / "huecos_clasificados.csv"
SALUD_PUBLICA_CSV = PROCESSED_DIR / "salud_cdmx_publico.csv"

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Procesados: {PROCESSED_DIR}")

## 1. Cargar o reconstruir los huecos significativos

Primero buscamos si el notebook 03 ya dejo una tabla de huecos. Si no existe, reconstruimos los huecos significativos desde `salud_cdmx_publico.csv` usando Gudhi, el mismo criterio del notebook 03: Alpha complex, pares de persistencia, circuncentro del triangulo que mata el ciclo y percentil 90 por nivel de atencion.

In [ ]:
def buscar_archivo_huecos():
    candidatos = [
        PROCESSED_DIR / "huecos_significativos.csv",
        PROCESSED_DIR / "huecos_h1_significativos.csv",
        PROCESSED_DIR / "huecos_publicos.csv",
        PROCESSED_DIR / "huecos_h1_publicos.csv",
    ]
    for path in candidatos:
        if path.exists():
            return path
    encontrados = sorted(PROCESSED_DIR.glob("*hueco*.csv"))
    return encontrados[0] if encontrados else None


def circuncentro(p1, p2, p3):
    ax, ay = p1
    bx, by = p2
    cx, cy = p3
    d = 2 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    if abs(d) < 1e-9:
        return (ax + bx + cx) / 3, (ay + by + cy) / 3
    a2 = ax * ax + ay * ay
    b2 = bx * bx + by * by
    c2 = cx * cx + cy * cy
    ux = (a2 * (by - cy) + b2 * (cy - ay) + c2 * (ay - by)) / d
    uy = (a2 * (cx - bx) + b2 * (ax - cx) + c2 * (bx - ax)) / d
    return ux, uy


def extraer_huecos_desde_alpha(puntos, nivel_atencion, percentil=90):
    import gudhi

    alpha = gudhi.AlphaComplex(points=puntos)
    st = alpha.create_simplex_tree()
    st.persistence()

    filas = []
    for s_birth, s_death in st.persistence_pairs():
        if len(s_birth) == 2 and len(s_death) == 3:
            birth = np.sqrt(st.filtration(s_birth))
            death = np.sqrt(st.filtration(s_death))
            tri = [alpha.get_point(i) for i in s_death]
            cx, cy = circuncentro(tri[0], tri[1], tri[2])
            filas.append({
                "nivel_atencion": nivel_atencion,
                "cx": cx,
                "cy": cy,
                "birth": birth,
                "death": death,
                "persistence": death - birth,
            })

    huecos = pd.DataFrame(filas)
    if huecos.empty:
        return huecos
    umbral = np.percentile(huecos["persistence"], percentil)
    return huecos[huecos["persistence"] >= umbral].copy()


def reconstruir_y_guardar_huecos():
    if not SALUD_PUBLICA_CSV.exists():
        raise FileNotFoundError(
            f"No existe {SALUD_PUBLICA_CSV}. Ejecuta antes el notebook 02."
        )

    df_pub = pd.read_csv(SALUD_PUBLICA_CSV)
    requeridas = {"nivel_atencion", "x_m", "y_m"}
    faltantes = requeridas - set(df_pub.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas en salud publica: {faltantes}")

    partes = []
    for nivel in ["primer_nivel", "segundo_tercer_nivel"]:
        sub = df_pub[df_pub["nivel_atencion"] == nivel].copy()
        pts = sub[["x_m", "y_m"]].to_numpy()
        print(f"Reconstruyendo huecos {nivel}: {len(pts):,} puntos")
        partes.append(extraer_huecos_desde_alpha(pts, nivel, percentil=90))

    huecos = pd.concat(partes, ignore_index=True)
    huecos = huecos.sort_values(["nivel_atencion", "persistence"], ascending=[True, False]).reset_index(drop=True)
    huecos.insert(0, "id_hueco", [f"H{i+1:03d}" for i in range(len(huecos))])
    huecos.to_csv(HUECOS_CSV, index=False)
    print(f"Huecos significativos guardados en {HUECOS_CSV} ({len(huecos):,} filas)")
    return HUECOS_CSV


huecos_path = buscar_archivo_huecos()
if huecos_path is None:
    print("No encontre tabla de huecos; la reconstruyo desde el notebook 02/03.")
    huecos_path = reconstruir_y_guardar_huecos()
else:
    print(f"Usando tabla de huecos existente: {huecos_path}")

In [ ]:
huecos = pd.read_csv(huecos_path)

renombres = {
    "r_nacimiento": "birth",
    "r_muerte": "death",
    "persistencia": "persistence",
}
huecos = huecos.rename(columns={k: v for k, v in renombres.items() if k in huecos.columns})

if "id_hueco" not in huecos.columns:
    huecos.insert(0, "id_hueco", [f"H{i+1:03d}" for i in range(len(huecos))])

columnas_minimas = ["id_hueco", "nivel_atencion", "cx", "cy", "birth", "death", "persistence"]
faltantes = [col for col in columnas_minimas if col not in huecos.columns]
if faltantes:
    raise ValueError(f"La tabla de huecos no tiene columnas minimas: {faltantes}")

huecos = huecos[columnas_minimas].copy()
for col in ["cx", "cy", "birth", "death", "persistence"]:
    huecos[col] = pd.to_numeric(huecos[col], errors="coerce")
huecos = huecos.dropna(subset=["cx", "cy", "death", "persistence"])

huecos.to_csv(HUECOS_CSV, index=False)
print(huecos.shape)
huecos.head()

## 2. Geometria circular de los huecos

Cada hueco se representa como un buffer circular alrededor de su centro topologico. Usamos como radio `death`, porque es el radio al que el triangulo rellena el ciclo. El area resultante se calcula en metros cuadrados y kilometros cuadrados.

In [ ]:
huecos_pts = gpd.GeoDataFrame(
    huecos,
    geometry=gpd.points_from_xy(huecos["cx"], huecos["cy"]),
    crs=CRS_METRICO,
)

huecos_gdf = huecos_pts.copy()
huecos_gdf["geometry"] = huecos_gdf.geometry.buffer(huecos_gdf["death"])
huecos_gdf["area_hueco_m2"] = huecos_gdf.geometry.area
huecos_gdf["area_hueco_km2"] = huecos_gdf["area_hueco_m2"] / 1_000_000

print(huecos_gdf.crs)
print(huecos_gdf[["id_hueco", "nivel_atencion", "death", "area_hueco_km2"]].head())

## 3. Poblacion por AGEB o manzana

La estimacion de poblacion dentro de cada hueco se hace por interseccion areal. Idealmente se usa AGEB o manzana con poblacion del Censo 2020. Si la capa ya trae `POBTOT`, `POB_TOT`, `pobtot` o una variante similar, el notebook la detecta. Si no, deja instrucciones para colocar una capa con poblacion en `data/external`.

Formula usada:

`poblacion_interseccion = poblacion_total_poligono * area_interseccion / area_poligono`

In [ ]:
def encontrar_geodata(patterns, directories=(EXTERNAL_DIR, RAW_DIR)):
    extensiones = ["*.shp", "*.gpkg", "*.geojson", "*.json"]
    paths = []
    for directory in directories:
        if not directory.exists():
            continue
        for ext in extensiones:
            paths.extend(directory.rglob(ext))
    paths = sorted(set(paths))
    for path in paths:
        name = path.name.lower()
        if all(pattern in name for pattern in patterns):
            return path
    return None


def elegir_columna(df, candidatos, contains=None, required=False):
    lower = {str(col).lower(): col for col in df.columns}
    for col in candidatos:
        if col in df.columns:
            return col
        if col.lower() in lower:
            return lower[col.lower()]
    if contains:
        for text in contains:
            for low, original in lower.items():
                if text in low:
                    return original
    if required:
        raise KeyError("No encontre columna esperada. Columnas disponibles: " + ", ".join(map(str, df.columns)))
    return None


def clean_code(series, width):
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(width)


def cargar_ageb_rezago_poblacion():
    path = encontrar_geodata(["ageb"], directories=(EXTERNAL_DIR, RAW_DIR))
    if path is None:
        print("No encontre capa AGEB/manzana. Coloca un SHP/GPKG/GeoJSON en data/external con poblacion Censo 2020.")
        return None, None

    gdf = gpd.read_file(path)
    print(f"Capa territorial cargada: {path}")
    print(f"Shape: {gdf.shape}")
    print("Columnas disponibles:", list(gdf.columns))

    cvegeo_col = elegir_columna(gdf, ["CVEGEO", "CVE_GEO", "CVE_AGEB"], contains=["cvegeo", "cve_geo"])
    ent_col = elegir_columna(gdf, ["CVE_ENT", "CVE_ENTI", "ENTIDAD", "clv_ntd"], contains=["cve_ent", "entidad", "clv_ntd"])
    if cvegeo_col is None and all(col in gdf.columns for col in ["clv_ntd", "clv_mnc", "clv_lcl", "ageb"]):
        cvegeo_col = "CVEGEO"
        gdf[cvegeo_col] = clean_code(gdf["clv_ntd"], 2) + clean_code(gdf["clv_mnc"], 3) + clean_code(gdf["clv_lcl"], 4) + clean_code(gdf["ageb"], 4)

    if ent_col is not None:
        gdf = gdf[clean_code(gdf[ent_col], 2) == "09"].copy()
    elif cvegeo_col is not None:
        gdf = gdf[gdf[cvegeo_col].astype(str).str.startswith("09")].copy()

    pob_col = elegir_columna(
        gdf,
        ["POBTOT", "POB_TOT", "pobtot", "pob_tot", "P_TOTAL", "POB_TOTAL", "PTOT"],
        contains=["pobtot", "pob_tot", "pob_total", "p_total"],
    )
    rezago_col = elegir_columna(
        gdf,
        ["GRS", "gdo_rezsoc", "grado_rezsoc", "GRADO_REZ", "GM_2020"],
        contains=["grs", "rezago", "grado"],
    )
    alcaldia_col = elegir_columna(
        gdf,
        ["NOMGEO", "NOM_MUN", "MUNICIPIO", "ALCALDIA", "nom_mun"],
        contains=["alcal", "municip", "nom_mun", "nomgeo"],
    )

    if gdf.crs is None:
        warnings.warn("La capa territorial no trae CRS. Se asumira EPSG:4326; verifica el archivo fuente.")
        gdf = gdf.set_crs(CRS_WEB)
    gdf = gdf.to_crs(CRS_METRICO)
    gdf["area_poligono_m2"] = gdf.geometry.area

    meta = {
        "path": path,
        "cvegeo_col": cvegeo_col,
        "pob_col": pob_col,
        "rezago_col": rezago_col,
        "alcaldia_col": alcaldia_col,
    }
    print("Columnas detectadas:", meta)
    return gdf, meta


ageb_gdf, ageb_meta = cargar_ageb_rezago_poblacion()

In [ ]:
def ponderar_poblacion_por_area(huecos_gdf, ageb_gdf, ageb_meta):
    base = pd.DataFrame({"id_hueco": huecos_gdf["id_hueco"]})
    if ageb_gdf is None or ageb_meta is None or ageb_meta.get("pob_col") is None:
        print("Sin columna de poblacion. Se dejan poblacion y densidad como 0 para no romper el flujo.")
        base["poblacion_estimada"] = 0.0
        base["densidad_pob_hab_km2"] = 0.0
        base["n_agebs_intersectadas"] = 0
        base["alcaldia_principal"] = pd.NA
        return base

    pob_col = ageb_meta["pob_col"]
    alcaldia_col = ageb_meta.get("alcaldia_col")
    cols = ["geometry", "area_poligono_m2", pob_col]
    if alcaldia_col:
        cols.append(alcaldia_col)

    inter = gpd.overlay(
        huecos_gdf[["id_hueco", "area_hueco_m2", "geometry"]],
        ageb_gdf[cols],
        how="intersection",
        keep_geom_type=False,
    )
    if inter.empty:
        base["poblacion_estimada"] = 0.0
        base["densidad_pob_hab_km2"] = 0.0
        base["n_agebs_intersectadas"] = 0
        base["alcaldia_principal"] = pd.NA
        return base

    inter["area_interseccion_m2"] = inter.geometry.area
    inter[pob_col] = pd.to_numeric(inter[pob_col], errors="coerce").fillna(0)
    inter["poblacion_interseccion"] = inter[pob_col] * (inter["area_interseccion_m2"] / inter["area_poligono_m2"].replace(0, np.nan))
    inter["poblacion_interseccion"] = inter["poblacion_interseccion"].fillna(0)

    resumen = inter.groupby("id_hueco").agg(
        poblacion_estimada=("poblacion_interseccion", "sum"),
        n_agebs_intersectadas=(pob_col, "size"),
    ).reset_index()

    area = huecos_gdf[["id_hueco", "area_hueco_km2"]]
    resumen = resumen.merge(area, on="id_hueco", how="left")
    resumen["densidad_pob_hab_km2"] = resumen["poblacion_estimada"] / resumen["area_hueco_km2"].replace(0, np.nan)

    if alcaldia_col:
        idx = inter.sort_values("area_interseccion_m2").groupby("id_hueco").tail(1)[["id_hueco", alcaldia_col]]
        idx = idx.rename(columns={alcaldia_col: "alcaldia_principal"})
        resumen = resumen.merge(idx, on="id_hueco", how="left")
    else:
        resumen["alcaldia_principal"] = pd.NA

    resumen = base.merge(resumen.drop(columns=["area_hueco_km2"], errors="ignore"), on="id_hueco", how="left")
    for col in ["poblacion_estimada", "densidad_pob_hab_km2", "n_agebs_intersectadas"]:
        resumen[col] = resumen[col].fillna(0)
    return resumen


pob_resumen = ponderar_poblacion_por_area(huecos_gdf, ageb_gdf, ageb_meta)
pob_resumen.head()

## 4. Rezago social o vulnerabilidad socioeconomica

Usamos rezago social como aproximacion academica a vulnerabilidad socio-territorial. No se interpreta como clase social directa. Si la capa trae una columna categorica de grado de rezago, la convertimos a escala 0-1. Si trae un indice numerico, tambien puede incorporarse como columna detectada o agregarse manualmente por `CVEGEO`.

Si no hay dato de rezago por AGEB, coloca un CSV en `data/external` con una clave geografica (`CVEGEO`) y variables como `grado_rezago` o `indice_rezago`.

In [ ]:
MAPA_REZAGO = {
    "muy bajo": 1,
    "bajo": 2,
    "medio": 3,
    "alto": 4,
    "muy alto": 5,
}


def score_rezago(valor):
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip().lower()
    if texto in MAPA_REZAGO:
        return MAPA_REZAGO[texto]
    try:
        return float(texto)
    except ValueError:
        return np.nan


def ponderar_rezago(huecos_gdf, ageb_gdf, ageb_meta):
    base = pd.DataFrame({"id_hueco": huecos_gdf["id_hueco"]})
    if ageb_gdf is None or ageb_meta is None or ageb_meta.get("rezago_col") is None:
        print("Sin columna de rezago. Se usa vulnerabilidad neutral = 0.5.")
        base["rezago_promedio_ponderado"] = np.nan
        base["grado_rezago_dominante"] = pd.NA
        base["pct_area_rezago_alto_muy_alto"] = np.nan
        base["indicador_socioeconomico_normalizado"] = 0.5
        return base

    rezago_col = ageb_meta["rezago_col"]
    cols = ["geometry", "area_poligono_m2", rezago_col]
    inter = gpd.overlay(
        huecos_gdf[["id_hueco", "geometry"]],
        ageb_gdf[cols],
        how="intersection",
        keep_geom_type=False,
    )
    if inter.empty:
        base["rezago_promedio_ponderado"] = np.nan
        base["grado_rezago_dominante"] = pd.NA
        base["pct_area_rezago_alto_muy_alto"] = np.nan
        base["indicador_socioeconomico_normalizado"] = 0.5
        return base

    inter["area_interseccion_m2"] = inter.geometry.area
    inter["rezago_score"] = inter[rezago_col].apply(score_rezago)
    inter["rezago_x_area"] = inter["rezago_score"] * inter["area_interseccion_m2"]
    inter["es_alto_muy_alto"] = inter[rezago_col].astype(str).str.lower().str.strip().isin(["alto", "muy alto"])

    resumen = inter.groupby("id_hueco").agg(
        area_total_intersectada=("area_interseccion_m2", "sum"),
        rezago_x_area=("rezago_x_area", "sum"),
        area_alto_muy_alto=("area_interseccion_m2", lambda s: s[inter.loc[s.index, "es_alto_muy_alto"]].sum()),
    ).reset_index()
    resumen["rezago_promedio_ponderado"] = resumen["rezago_x_area"] / resumen["area_total_intersectada"].replace(0, np.nan)
    resumen["pct_area_rezago_alto_muy_alto"] = resumen["area_alto_muy_alto"] / resumen["area_total_intersectada"].replace(0, np.nan)

    dom = inter.sort_values("area_interseccion_m2").groupby("id_hueco").tail(1)[["id_hueco", rezago_col]]
    dom = dom.rename(columns={rezago_col: "grado_rezago_dominante"})
    resumen = resumen.merge(dom, on="id_hueco", how="left")

    min_val = resumen["rezago_promedio_ponderado"].min()
    max_val = resumen["rezago_promedio_ponderado"].max()
    if pd.isna(min_val) or pd.isna(max_val) or min_val == max_val:
        resumen["indicador_socioeconomico_normalizado"] = 0.5
    else:
        resumen["indicador_socioeconomico_normalizado"] = (resumen["rezago_promedio_ponderado"] - min_val) / (max_val - min_val)

    cols_finales = [
        "id_hueco", "rezago_promedio_ponderado", "grado_rezago_dominante",
        "pct_area_rezago_alto_muy_alto", "indicador_socioeconomico_normalizado",
    ]
    resumen = base.merge(resumen[cols_finales], on="id_hueco", how="left")
    resumen["indicador_socioeconomico_normalizado"] = resumen["indicador_socioeconomico_normalizado"].fillna(0.5)
    return resumen


rezago_resumen = ponderar_rezago(huecos_gdf, ageb_gdf, ageb_meta)
rezago_resumen.head()

## 5. Areas verdes, parques y usos no habitacionales

Este paso descuenta huecos que son topologicamente reales pero territorialmente esperables: parques, suelo de conservacion, cuerpos de agua, aeropuerto, deportivos, panteones, campus grandes u otros usos no habitacionales.

Pipeline esperado: colocar una o varias capas poligonales en `data/external/no_habitacional/` o `data/raw/` con nombres que incluyan `parque`, `area_verde`, `suelo_conservacion`, `agua`, `aeropuerto`, `panteon`, `deportivo`, `campus` o `no_habitacional`. Si tienen una columna de tipo/etiqueta, el notebook intenta detectarla.

In [ ]:
def encontrar_capas_no_habitacionales():
    keywords = [
        "parque", "area_verde", "areas_verdes", "verde", "conservacion",
        "agua", "aeropuerto", "panteon", "deportivo", "campus", "no_habitacional",
    ]
    extensiones = ["*.shp", "*.gpkg", "*.geojson", "*.json"]
    paths = []
    for directory in [EXTERNAL_DIR / "no_habitacional", EXTERNAL_DIR, RAW_DIR]:
        if not directory.exists():
            continue
        for ext in extensiones:
            for path in directory.rglob(ext):
                name = path.name.lower()
                if any(keyword in name for keyword in keywords):
                    paths.append(path)
    return sorted(set(paths))


def cargar_no_habitacional():
    paths = encontrar_capas_no_habitacionales()
    if not paths:
        print("No encontre capas no habitacionales. Se usara pct_no_habitacional = 0 como respaldo temporal.")
        return None

    partes = []
    for path in paths:
        try:
            gdf = gpd.read_file(path)
            if gdf.empty:
                continue
            if gdf.crs is None:
                warnings.warn(f"{path} no trae CRS. Se asumira EPSG:4326.")
                gdf = gdf.set_crs(CRS_WEB)
            gdf = gdf.to_crs(CRS_METRICO)
            etiqueta_col = elegir_columna(
                gdf,
                ["tipo", "TIPO", "categoria", "CATEGORIA", "nombre", "NOMBRE"],
                contains=["tipo", "categoria", "nombre"],
            )
            gdf["fuente_no_habitacional"] = path.stem
            if etiqueta_col:
                gdf["etiqueta_no_habitacional"] = gdf[etiqueta_col].astype(str)
            else:
                gdf["etiqueta_no_habitacional"] = path.stem
            partes.append(gdf[["fuente_no_habitacional", "etiqueta_no_habitacional", "geometry"]])
            print(f"Capa no habitacional cargada: {path} ({len(gdf):,} poligonos)")
        except Exception as exc:
            warnings.warn(f"No pude cargar {path}: {exc}")

    if not partes:
        return None
    return pd.concat(partes, ignore_index=True).pipe(gpd.GeoDataFrame, geometry="geometry", crs=CRS_METRICO)


def calcular_no_habitacional(huecos_gdf, nohab_gdf):
    base = huecos_gdf[["id_hueco", "area_hueco_m2"]].copy()
    if nohab_gdf is None or nohab_gdf.empty:
        base["area_no_habitacional_m2"] = 0.0
        base["pct_no_habitacional"] = 0.0
        base["pct_area_verde"] = 0.0
        base["etiqueta_no_habitacional_dominante"] = pd.NA
        return base.drop(columns=["area_hueco_m2"])

    inter = gpd.overlay(
        huecos_gdf[["id_hueco", "area_hueco_m2", "geometry"]],
        nohab_gdf[["etiqueta_no_habitacional", "geometry"]],
        how="intersection",
        keep_geom_type=False,
    )
    if inter.empty:
        base["area_no_habitacional_m2"] = 0.0
        base["pct_no_habitacional"] = 0.0
        base["pct_area_verde"] = 0.0
        base["etiqueta_no_habitacional_dominante"] = pd.NA
        return base.drop(columns=["area_hueco_m2"])

    inter["area_interseccion_m2"] = inter.geometry.area
    inter["es_area_verde"] = inter["etiqueta_no_habitacional"].astype(str).str.lower().str.contains("verde|parque|conservacion")

    resumen = inter.groupby("id_hueco").agg(
        area_no_habitacional_m2=("area_interseccion_m2", "sum"),
        area_verde_m2=("area_interseccion_m2", lambda s: s[inter.loc[s.index, "es_area_verde"]].sum()),
    ).reset_index()
    dom = inter.sort_values("area_interseccion_m2").groupby("id_hueco").tail(1)[["id_hueco", "etiqueta_no_habitacional"]]
    dom = dom.rename(columns={"etiqueta_no_habitacional": "etiqueta_no_habitacional_dominante"})
    resumen = resumen.merge(dom, on="id_hueco", how="left")
    resumen = base.merge(resumen, on="id_hueco", how="left")
    resumen[["area_no_habitacional_m2", "area_verde_m2"]] = resumen[["area_no_habitacional_m2", "area_verde_m2"]].fillna(0)
    resumen["pct_no_habitacional"] = (resumen["area_no_habitacional_m2"] / resumen["area_hueco_m2"].replace(0, np.nan)).clip(0, 1).fillna(0)
    resumen["pct_area_verde"] = (resumen["area_verde_m2"] / resumen["area_hueco_m2"].replace(0, np.nan)).clip(0, 1).fillna(0)
    return resumen.drop(columns=["area_hueco_m2", "area_verde_m2"])


nohab_gdf = cargar_no_habitacional()
nohab_resumen = calcular_no_habitacional(huecos_gdf, nohab_gdf)
nohab_resumen.head()

## 6. Score de prioridad y clasificacion automatica

Creamos dos scores. El multiplicativo es estricto: un hueco necesita persistencia, poblacion, vulnerabilidad y bajo porcentaje no habitacional para subir. El aditivo permite ordenar casos intermedios aunque una variable sea baja.

La clasificacion automatica usa reglas claras:

- `Hueco esperado / baja prioridad`: mucho uso no habitacional o poblacion estimada muy baja.
- `Prioritario`: score en el top 25% y bajo porcentaje no habitacional.
- `Geometricamente importante, territorialmente dudoso`: alta persistencia, pero alto porcentaje no habitacional.
- `Revision manual`: zona intermedia.

In [ ]:
def normalizar_01(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series(0.5, index=series.index)
    return ((series - min_val) / (max_val - min_val)).clip(0, 1)


tabla = huecos_gdf.drop(columns="geometry").merge(pob_resumen, on="id_hueco", how="left")
tabla = tabla.merge(rezago_resumen, on="id_hueco", how="left")
tabla = tabla.merge(nohab_resumen, on="id_hueco", how="left")

rellenos = {
    "poblacion_estimada": 0,
    "densidad_pob_hab_km2": 0,
    "n_agebs_intersectadas": 0,
    "pct_no_habitacional": 0,
    "pct_area_verde": 0,
    "area_no_habitacional_m2": 0,
    "indicador_socioeconomico_normalizado": 0.5,
}
for col, value in rellenos.items():
    if col not in tabla.columns:
        tabla[col] = value
    tabla[col] = tabla[col].fillna(value)

tabla["persistence_norm"] = normalizar_01(tabla["persistence"])
tabla["poblacion_norm"] = normalizar_01(tabla["poblacion_estimada"])
tabla["densidad_norm"] = normalizar_01(tabla["densidad_pob_hab_km2"])
tabla["vulnerabilidad_norm"] = tabla["indicador_socioeconomico_normalizado"].clip(0, 1)

tabla["priority_score"] = (
    tabla["persistence_norm"]
    * tabla["poblacion_norm"]
    * tabla["vulnerabilidad_norm"]
    * (1 - tabla["pct_no_habitacional"].clip(0, 1))
)
tabla["priority_score_aditivo"] = (
    0.30 * tabla["persistence_norm"]
    + 0.25 * tabla["poblacion_norm"]
    + 0.25 * tabla["densidad_norm"]
    + 0.20 * tabla["vulnerabilidad_norm"]
    - 0.30 * tabla["pct_no_habitacional"].clip(0, 1)
).clip(lower=0, upper=1)

umbral_prioridad = tabla["priority_score_aditivo"].quantile(QUANTILE_PRIORIDAD)
umbral_persistencia_alta = tabla["persistence_norm"].quantile(0.75)

def clasificar(row):
    if row["pct_no_habitacional"] > PCT_NO_HABITACIONAL_UMBRAL or row["poblacion_estimada"] < POBLACION_BAJA_UMBRAL:
        if row["persistence_norm"] >= umbral_persistencia_alta and row["pct_no_habitacional"] > PCT_NO_HAB_DUDOSO:
            return "Geometricamente importante, territorialmente dudoso"
        return "Hueco esperado / baja prioridad"
    if row["priority_score_aditivo"] >= umbral_prioridad and row["pct_no_habitacional"] < PCT_NO_HAB_DUDOSO:
        return "Prioritario"
    return "Revision manual"


tabla["clasificacion"] = tabla.apply(clasificar, axis=1)

columnas_finales = [
    "id_hueco", "nivel_atencion", "cx", "cy", "birth", "death", "persistence",
    "area_hueco_km2", "poblacion_estimada", "densidad_pob_hab_km2",
    "n_agebs_intersectadas", "alcaldia_principal", "rezago_promedio_ponderado",
    "grado_rezago_dominante", "pct_area_rezago_alto_muy_alto", "pct_no_habitacional",
    "pct_area_verde", "etiqueta_no_habitacional_dominante", "persistence_norm",
    "poblacion_norm", "densidad_norm", "vulnerabilidad_norm", "priority_score",
    "priority_score_aditivo", "clasificacion",
]
for col in columnas_finales:
    if col not in tabla.columns:
        tabla[col] = pd.NA

tabla_final = tabla[columnas_finales].sort_values("priority_score_aditivo", ascending=False).reset_index(drop=True)
tabla_final.to_csv(HUECOS_CLASIFICADOS_CSV, index=False)

print(f"Tabla final guardada en {HUECOS_CLASIFICADOS_CSV}")
print(tabla_final["clasificacion"].value_counts())
tabla_final.head(10)

## 7. Mapas estaticos

Los mapas se exportan a `reports/figures/`. Sirven para revisar si la clasificacion automatica tiene sentido espacialmente antes de usar el mapa interactivo.

In [ ]:
huecos_plot = huecos_gdf.merge(tabla_final[["id_hueco", "priority_score_aditivo", "clasificacion"]], on="id_hueco", how="left")

COLORES_CLASIFICACION = {
    "Prioritario": "#d73027",
    "Revision manual": "#fc8d59",
    "Hueco esperado / baja prioridad": "#91cf60",
    "Geometricamente importante, territorialmente dudoso": "#756bb1",
}


def plot_base(ax, titulo):
    if ageb_gdf is not None:
        ageb_gdf.boundary.plot(ax=ax, color="#d0d0d0", linewidth=0.25)
    ax.set_title(titulo)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m, UTM 14N)")
    ax.set_ylabel("y (m, UTM 14N)")


fig, ax = plt.subplots(figsize=(9, 9))
plot_base(ax, "Huecos H1 clasificados")
for clas, color in COLORES_CLASIFICACION.items():
    sub = huecos_plot[huecos_plot["clasificacion"] == clas]
    if not sub.empty:
        sub.boundary.plot(ax=ax, color=color, linewidth=1.8, label=clas)
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mapa_huecos_clasificacion.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(9, 9))
plot_base(ax, "Huecos H1 por score de prioridad")
huecos_plot.plot(ax=ax, column="priority_score_aditivo", cmap="YlOrRd", alpha=0.55, edgecolor="#333333", linewidth=0.7, legend=True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mapa_huecos_prioridad.png", dpi=200)
plt.show()

In [ ]:
if ageb_gdf is not None and ageb_meta is not None and ageb_meta.get("pob_col") is not None:
    ageb_pob = ageb_gdf.copy()
    pob_col = ageb_meta["pob_col"]
    ageb_pob[pob_col] = pd.to_numeric(ageb_pob[pob_col], errors="coerce").fillna(0)
    ageb_pob["densidad_hab_km2"] = ageb_pob[pob_col] / (ageb_pob["area_poligono_m2"] / 1_000_000).replace(0, np.nan)
    fig, ax = plt.subplots(figsize=(9, 9))
    ageb_pob.plot(ax=ax, column="densidad_hab_km2", cmap="Blues", legend=True, alpha=0.75, linewidth=0)
    huecos_plot.boundary.plot(ax=ax, color="#d73027", linewidth=1.2)
    ax.set_title("Huecos sobre densidad poblacional")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "mapa_huecos_densidad_poblacional.png", dpi=200)
    plt.show()

if ageb_gdf is not None and ageb_meta is not None and ageb_meta.get("rezago_col") is not None:
    rezago_col = ageb_meta["rezago_col"]
    ageb_rez = ageb_gdf.copy()
    ageb_rez["rezago_score"] = ageb_rez[rezago_col].apply(score_rezago)
    fig, ax = plt.subplots(figsize=(9, 9))
    ageb_rez.plot(ax=ax, column="rezago_score", cmap="OrRd", legend=True, alpha=0.75, linewidth=0)
    huecos_plot.boundary.plot(ax=ax, color="#252525", linewidth=1.1)
    ax.set_title("Huecos sobre rezago social")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "mapa_huecos_rezago_social.png", dpi=200)
    plt.show()

fig, ax = plt.subplots(figsize=(9, 9))
plot_base(ax, "Huecos sobre areas no habitacionales")
if nohab_gdf is not None:
    nohab_gdf.plot(ax=ax, color="#31a354", alpha=0.45, linewidth=0)
huecos_plot.boundary.plot(ax=ax, color="#d73027", linewidth=1.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mapa_huecos_no_habitacional.png", dpi=200)
plt.show()

## 8. Mapa HTML interactivo

El mapa interactivo se genera en EPSG:4326 para Leaflet/Folium. Incluye huecos por clasificacion, puntos de salud publica por nivel de atencion y, si esta disponible, una capa de areas no habitacionales.

In [ ]:
def crear_mapa_interactivo():
    if folium is None:
        print("Folium no esta instalado. Instala folium para crear el mapa HTML.")
        return

    huecos_web = huecos_plot.merge(
        tabla_final.drop(columns=["nivel_atencion", "cx", "cy", "birth", "death", "persistence"], errors="ignore"),
        on="id_hueco",
        how="left",
        suffixes=("", "_tabla"),
    ).to_crs(CRS_WEB)
    # Folium/Leaflet es sensible a pd.NA, NaN y tipos numpy dentro de GeoJSON.
    # Sanitizamos propiedades antes de renderizar para evitar HTML vacios si falla m.save().
    for col in huecos_web.columns:
        if col == "geometry":
            continue
        if pd.api.types.is_numeric_dtype(huecos_web[col]):
            huecos_web[col] = pd.to_numeric(huecos_web[col], errors="coerce").fillna(0)
        else:
            huecos_web[col] = huecos_web[col].astype(object).where(pd.notna(huecos_web[col]), "")

    centro = huecos_web.geometry.centroid
    lat = centro.y.mean() if len(centro) else 19.4326
    lon = centro.x.mean() if len(centro) else -99.1332
    m = folium.Map(location=[lat, lon], zoom_start=11, tiles="CartoDB positron", prefer_canvas=True)

    if ageb_gdf is not None and ageb_meta is not None and ageb_meta.get("rezago_col") is not None:
        ageb_web = ageb_gdf[[ageb_meta["rezago_col"], "geometry"]].copy().to_crs(CRS_WEB)
        rezago_col = ageb_meta["rezago_col"]
        ageb_web[rezago_col] = ageb_web[rezago_col].astype(object).where(pd.notna(ageb_web[rezago_col]), "Sin dato")
        capa_rezago = folium.FeatureGroup(name="Rezago social / vulnerabilidad", show=False)
        folium.GeoJson(
            ageb_web,
            style_function=lambda feature: {"fillColor": "#fdae61", "color": "#777777", "weight": 0.2, "fillOpacity": 0.25},
            tooltip=folium.GeoJsonTooltip(fields=[rezago_col], aliases=["Rezago"]),
        ).add_to(capa_rezago)
        capa_rezago.add_to(m)

    if nohab_gdf is not None:
        nohab_web = nohab_gdf[["etiqueta_no_habitacional", "geometry"]].copy().to_crs(CRS_WEB)
        nohab_web["etiqueta_no_habitacional"] = nohab_web["etiqueta_no_habitacional"].astype(object).where(pd.notna(nohab_web["etiqueta_no_habitacional"]), "No habitacional")
        capa_nohab = folium.FeatureGroup(name="Areas no habitacionales", show=False)
        folium.GeoJson(
            nohab_web,
            style_function=lambda feature: {"fillColor": "#31a354", "color": "#238b45", "weight": 0.4, "fillOpacity": 0.35},
            tooltip=folium.GeoJsonTooltip(fields=["etiqueta_no_habitacional"], aliases=["Uso"]),
        ).add_to(capa_nohab)
        capa_nohab.add_to(m)

    df_pub = pd.read_csv(SALUD_PUBLICA_CSV)
    colores_nivel = {"primer_nivel": "#2c7fb8", "segundo_tercer_nivel": "#d62728"}
    for nivel, color in colores_nivel.items():
        capa = folium.FeatureGroup(name=f"Salud publica - {nivel}", show=True)
        sub = df_pub[df_pub["nivel_atencion"] == nivel]
        for _, row in sub.iterrows():
            folium.CircleMarker(
                location=[row["latitud"], row["longitud"]],
                radius=3 + np.log2(max(row.get("n_servicios", 1), 1)),
                color="#ffffff",
                weight=0.7,
                fill=True,
                fill_color=color,
                fill_opacity=0.8,
                tooltip=f"{row.get('nom_estab', '')}<br>{nivel}<br>Servicios: {row.get('n_servicios', 1)}",
            ).add_to(capa)
        capa.add_to(m)

    for clasificacion, color in COLORES_CLASIFICACION.items():
        capa = folium.FeatureGroup(name=f"Huecos - {clasificacion}", show=True)
        sub = huecos_web[huecos_web["clasificacion"] == clasificacion]
        for _, row in sub.iterrows():
            popup_html = f"""
            <b>{row['id_hueco']}</b><br>
            Nivel: {row['nivel_atencion']}<br>
            Persistencia: {row['persistence']:.0f} m<br>
            Radio death: {row['death']:.0f} m<br>
            Poblacion estimada: {row.get('poblacion_estimada', 0):.0f}<br>
            Densidad: {row.get('densidad_pob_hab_km2', 0):.0f} hab/km2<br>
            Rezago dominante: {row.get('grado_rezago_dominante', 'NA')}<br>
            No habitacional: {100 * row.get('pct_no_habitacional', 0):.1f}%<br>
            Score: {row.get('priority_score_aditivo', 0):.3f}<br>
            Clasificacion: {row['clasificacion']}
            """
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda feature, color=color: {"fillColor": color, "color": color, "weight": 2, "fillOpacity": 0.18},
                tooltip=f"{row['id_hueco']} - {clasificacion}",
                popup=folium.Popup(popup_html, max_width=360),
            ).add_to(capa)
        capa.add_to(m)

    legend = """
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white; padding: 12px; border: 1px solid #777; border-radius: 6px; font-family: Arial; font-size: 12px;">
    <b>Clasificacion de huecos</b><br>
    <span style="color:#d73027;">●</span> Prioritario<br>
    <span style="color:#fc8d59;">●</span> Revision manual<br>
    <span style="color:#91cf60;">●</span> Baja prioridad / esperado<br>
    <span style="color:#756bb1;">●</span> Geometricamente importante, dudoso
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend))
    folium.LayerControl(collapsed=False).add_to(m)
    output = MAPS_DIR / "huecos_clasificados_interactivo.html"
    tmp_output = output.with_suffix(".tmp.html")
    m.save(tmp_output)
    if tmp_output.stat().st_size == 0:
        raise RuntimeError("Folium genero un HTML vacio; revisa las capas GeoJSON.")
    tmp_output.replace(output)
    print(f"Mapa interactivo guardado en {output} ({output.stat().st_size / 1_000_000:.2f} MB)")


crear_mapa_interactivo()

## 9. Conclusiones automaticas

Esta seccion resume la clasificacion. La interpretacion sustantiva debe revisarse con el mapa, porque un hueco puede ser persistente y aun asi corresponder a una zona no habitacional, un parque o un equipamiento urbano donde no se espera una unidad de salud.

In [ ]:
conteos = tabla_final["clasificacion"].value_counts()
n_prioritarios = int(conteos.get("Prioritario", 0))
n_baja = int(conteos.get("Hueco esperado / baja prioridad", 0))
n_revision = int(conteos.get("Revision manual", 0))
n_dudoso = int(conteos.get("Geometricamente importante, territorialmente dudoso", 0))

print("Resumen de clasificacion")
print(f"Huecos prioritarios: {n_prioritarios}")
print(f"Huecos de baja prioridad / esperados: {n_baja}")
print(f"Huecos en revision manual: {n_revision}")
print(f"Huecos geometricamente importantes pero dudosos: {n_dudoso}")

print("\nTop 10 huecos por priority_score_aditivo")
cols_top = [
    "id_hueco", "nivel_atencion", "alcaldia_principal", "persistence", "death",
    "poblacion_estimada", "densidad_pob_hab_km2", "grado_rezago_dominante",
    "pct_no_habitacional", "priority_score_aditivo", "clasificacion",
]
display(tabla_final[cols_top].head(10))

if n_prioritarios > 0:
    prioritarios = tabla_final[tabla_final["clasificacion"] == "Prioritario"]
    print("\nLectura preliminar:")
    print(
        "Los huecos prioritarios son candidatos donde coinciden robustez geometrica, "
        "poblacion estimada, vulnerabilidad socio-territorial y bajo porcentaje de uso no habitacional. "
        "Deben interpretarse como zonas candidatas para revision de cobertura publica, no como diagnostico definitivo."
    )
    if "alcaldia_principal" in prioritarios.columns:
        print("\nAlcaldias principales entre prioritarios:")
        print(prioritarios["alcaldia_principal"].value_counts().head(10))
else:
    print("\nNo se clasificaron huecos como prioritarios con los umbrales actuales. Revisa capas de poblacion/no habitacional y, si procede, ajusta umbrales.")

## Cierre metodologico

Los huecos H1 detectados por persistencia homologica son candidatos geometricos a vacios de cobertura. La persistencia indica que el hueco es robusto dentro de la nube de puntos, pero no mide directamente necesidad social, demanda sanitaria ni poblacion afectada.

Para reducir falsos positivos, este notebook cruza los huecos con poblacion, rezago social y usos no habitacionales. Un hueco sobre un parque, suelo de conservacion o zona no poblada puede ser topologicamente real pero no necesariamente prioritario. En cambio, un hueco persistente sobre una zona poblada y vulnerable puede interpretarse como candidato fuerte para intervencion publica.

El resultado no reemplaza evaluacion urbana, sanitaria ni trabajo de campo. Sirve como herramienta de priorizacion territorial para decidir que huecos merecen revision institucional y analisis mas fino.